In [ ]:
# # 01 - Exploratory Data Analysis
# PizzaHub sells one pizza ($16) across four regions. On 1 Mar 2023 a 5% **food**
# discount started in Region 3 and a 5% **delivery-fee** discount in Region 4.
# Regions 1-2 received nothing and act as controls.
#
# **Note:** the dataset is synthetic (region-specific means and volatilities).

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
from src.data import ROOT, load_orders, POLICY_DATE

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": .3})
FIG = ROOT / "figures"
df = load_orders()
df.head()

In [ ]:
# ## Data quality

print(df.shape, "| date range:", df.date.min().date(), "to", df.date.max().date())
print("Missing values:", int(df.isna().sum().sum()))
print("Rows per region:", df.groupby("region").size().to_dict())
print("Min daily orders:", df.qty.min())
df.groupby("region")[["price", "fee", "distance"]].agg(["min", "max"])

In [ ]:
# Prices and fees change only where expected: price in Region 3, fee in Region 4.
# Quantities are strictly positive, so a log model is valid.

In [ ]:
# ## Pre vs post orders by region

summary = df.groupby(["region", "post"]).qty.agg(["mean", "std"]).unstack()
summary.columns = ["mean pre", "mean post", "sd pre", "sd post"]
summary["change %"] = (summary["mean post"] / summary["mean pre"] - 1) * 100
summary.round(2)

ax = summary[["mean pre", "mean post"]].plot.bar(rot=0, width=.75)
ax.set_xlabel("Region"); ax.set_ylabel("Avg daily orders")
ax.set_title("Average daily orders: before vs after 1 Mar 2023")
plt.tight_layout(); plt.savefig(FIG / "pre_post_orders.png", dpi=150); plt.show()

In [ ]:
# Controls (1-2) drift down slightly, while treated regions (3-4) rise sharply.
# Note that treated regions are also noisier before the policy (higher sd), so
# robust standard errors matter in the next notebook.

In [ ]:
# ## Weekly trend

weekly = (df.assign(week=df["date"].dt.to_period("W").dt.start_time)
            .groupby(["week", "region"]).qty.mean().unstack())
ax = weekly.plot(marker="o", ms=3)
ax.axvline(POLICY_DATE, ls=":", color="k", label="policy start")
ax.set_ylabel("Avg daily orders"); ax.set_title("Weekly average orders by region")
ax.legend(title="Region")
plt.tight_layout(); plt.savefig(FIG / "weekly_orders.png", dpi=150); plt.show()


In [ ]:
# **Takeaway:** the treated regions step up after the policy date and the controls
# don't. Next: quantify this with difference-in-differences and test whether the
# comparison is credible.